In [1]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-text-splitters langchain-core
!pip install -q pypdf sentence-transformers faiss-cpu
!pip install -q pandas openpyxl tabulate!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-text-splitters langchain-core
!pip install -q pypdf sentence-transformers faiss-cpu
!pip install -q pandas openpyxl tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.9 MB/s eta 0:00:00
ERROR: Invalid requirement: 'tabulate!pip': Expected end or semicolon (after name and no valid version specifier)
    tabulate!pip
            ^


In [4]:
import os
import json
import re
import torch
import pandas as pd
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict
import warnings

warnings.filterwarnings('ignore')

from pypdf import PdfReader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document


# ============================================================================
# 2. データ構造の定義
# ============================================================================
@dataclass
class FinancialData:
    年度: str
    営業収益: str = "取得失敗"
    税引前利益: str = "取得失敗"
    当期純利益: str = "取得失敗"
    総資産: str = "取得失敗"
    ROE: str = "取得失敗"

    def to_dict(self):
        return asdict(self)


# ============================================================================
# 3. 正確な数値抽出クラス
# ============================================================================
class AccurateNumericalExtractor:

    def _get_value_for_column(
        self,
        text_block: str,
        keyword: str,
        unit_regex: str,
        target_column: int
    ) -> Optional[str]:

        next_metric_keywords = [
            "営業収益", "税引前利益", "親会社の所有者に帰属する当期利益",
            "資産合計", "純資産合計", "親会社の所有者に帰属する持分当期利益率",
            "総資産経常利益率", "自己資本比率", "一株当たり当期純利益",
            "一株当たり純資産額", "従業員数"
        ]

        next_metric_pattern = '|'.join(
            [re.escape(k) for k in next_metric_keywords if k != keyword]
        )

        pattern = (
            rf'^{re.escape(keyword)}\s*{re.escape(unit_regex)}\s*'
            rf'(.*?)(?=\n|{next_metric_pattern}|\Z)'
        )

        match = re.search(pattern, text_block, re.MULTILINE)

        if not match:
            fallback = (
                rf'^{re.escape(keyword)}.*?(\s+.*?)?'
                rf'(?=\n|{next_metric_pattern}|\Z)'
            )
            match = re.search(fallback, text_block, re.MULTILINE)

        if match:
            values_str = match.group(1).strip() if match.group(1) else ""
            all_raw_values = re.findall(r'[\d,]+(?:\.\d+)?|－', values_str)
            numeric_values = [v for v in all_raw_values if v != '－']

            if len(numeric_values) >= target_column:
                return numeric_values[target_column - 1]

        return None

    def extract_2025_data(self, pdf_path: str) -> FinancialData:
        reader = PdfReader(pdf_path)
        page5_text = reader.pages[4].extract_text()

        data = FinancialData(年度="2025年3月期")
        column_index = 5  # 2025年3月期は右端列

        mapping = [
            ("営業収益", "営業収益", "（百万円）"),
            ("税引前利益", "税引前利益", "（百万円）"),
            ("親会社の所有者に帰属する当期利益", "当期純利益", "（百万円）"),
            ("資産合計", "総資産", "（百万円）"),
            ("親会社の所有者に帰属する持分当期利益率", "ROE", "（％）")
        ]

        for jp_keyword, attr_name, unit_str in mapping:
            value = self._get_value_for_column(page5_text, jp_keyword, unit_str, column_index)

            if value:
                if unit_str == "（％）":
                    setattr(data, attr_name, f"{value}%")
                else:
                    setattr(data, attr_name, f"{value} 百万円")

        return data


# ============================================================================
# 4. RAG & Llama 要約クラス
# ============================================================================
class HybridAnalyzer:

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.full_text = ""
        self.numerical_data = None
        self.vectorstore = None
        self.summarizer_pipeline = None

    def prepare(self):
        print("1/4: 数値データを抽出中...")
        extractor = AccurateNumericalExtractor()
        self.numerical_data = extractor.extract_2025_data(self.pdf_path)

        print("2/4: 全文テキストを読み込み中（50ページ）...")
        reader = PdfReader(self.pdf_path)
        for i in range(min(50, len(reader.pages))):
            self.full_text += reader.pages[i].extract_text() + "\n"

        print("3/4: RAGデータベースを構築中...")
        embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-small")
        splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
        docs = [Document(page_content=t) for t in splitter.split_text(self.full_text)]
        self.vectorstore = FAISS.from_documents(docs, embeddings)

        print("4/4: Llama-3.2-3B モデルをロード中...")
        model_id = "meta-llama/Llama-3.2-3B-Instruct"

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )

        self.summarizer_pipeline = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer
        )

        print("--- 全準備完了 ---")

    def analyze(self, query: str):
        search_results = self.vectorstore.similarity_search(query, k=4)
        context = "\n\n".join([d.page_content for d in search_results])

        num_info = json.dumps(self.numerical_data.to_dict(), ensure_ascii=False, indent=2)

        prompt = f"""
あなたはトヨタ自動車の専門アナリストです。
提供された【正確な数値データ】を基軸とし、【参考文書】からその背景や要因を読み取って要約してください。

### 指示:
1. 【正確な数値データ】にある数値は一字一句変えずに使用すること。
2. なぜそのような業績になったのか、参考文書から「要因」や「今後の展望」を抽出すること。
3. 箇条書きで分かりやすく日本語で出力すること。

【正確な数値データ】
{num_info}

【参考文書】
{context}

要約：
"""

        outputs = self.summarizer_pipeline(
            [{"role": "user", "content": prompt}],
            max_new_tokens=800,
            temperature=0.2
        )

        return outputs[0]["generated_text"][-1]["content"]


# ============================================================================
# 5. 実行
# ============================================================================
def main():
    from google.colab import files

    print("トヨタ自動車の有価証券報告書（PDF）をアップロードしてください。")
    uploaded = files.upload()
    pdf_file = list(uploaded.keys())[0]

    analyzer = HybridAnalyzer(pdf_file)
    analyzer.prepare()

    question = "2025年3月期の業績ハイライトと、その主な要因、今後の見通しを教えてください。"
    result = analyzer.analyze(question)

    print("\n" + "=" * 30 + " 分析結果 " + "=" * 30)
    print(result)
    print("=" * 68)
    print("\n✅ すべて完了！")


if __name__ == "__main__":
    main()

トヨタ自動車の有価証券報告書（PDF）をアップロードしてください。


Saving トヨタ_2025_03_有価証券報告書.pdf to トヨタ_2025_03_有価証券報告書 (2).pdf
1/4: 数値データを抽出中...
2/4: 全文テキストを読み込み中（50ページ）...
3/4: RAGデータベースを構築中...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4/4: Llama-3.2-3B モデルをロード中...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- 全準備完了 ---

============================== 分析結果 ==============================
**トヨタの2025年3月期の業績**

*  営業収益：48兆367億円（前期比増減2兆9,413億円、6.5%）
*  営業利益：４兆7,955億円（前期比増減△5,573億円、△10.4%）
*  税引前利益：６兆4,145億円（前期比増減△5,504億円、△7.9%）
*  当期利益：４兆7,650億円（前期比増減△1,798億円、△3.6%）

**業績の主な増減要因**

*  営業面の努力：1,450億円
*  原価改善の努力：±0億円
*  為替変動の影響：5,900億円
*  諸経費の増減・低減努力：△9,900億円
*  その他：△3,023億円

**事業別の業績**

*  自動車事業：営業収益43兆1,998億円（前連結会計年度に比べて1兆9,336億円、4.7%の増収）、営業利益３兆9,402億円（前連結会計年度に比べて6,811億円、14.7%の減益）
*  金融事業：営業収益未記載

**トヨタのリスク**

*  自動車市場の競争激化
*   世界の自動車産業におけるCASEなどの技術革新
*   各国の税制優遇措置

**トヨタの人材育成と労使**

*   育成を含む人への投資について、継続的な対話を続けてきています
*   育成を含む人への投資について、労使間で議論を実施しています
*   育成を含む人への投資について、スピーディな変革に繋がるよう具体的な取り組みまで確認しています

✅ すべて完了！


In [3]:
import os
import json
import re
import torch
import pandas as pd
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
import warnings
warnings.filterwarnings('ignore')

# PDF処理
from pypdf import PdfReader

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

# RAG関連
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# ============================================================================
# 3. データクラスの定義
# ============================================================================

@dataclass
class FinancialMetrics:
    """財務指標を格納するデータクラス"""
    営業収益: Optional[str] = None
    税引前利益: Optional[str] = None
    当期純利益: Optional[str] = None
    総資産: Optional[str] = None
    純資産: Optional[str] = None
    売上高: Optional[str] = None
    経常利益: Optional[str] = None
    ROE: Optional[str] = None
    ROA: Optional[str] = None
    自己資本比率: Optional[str] = None
    従業員数: Optional[str] = None

    def to_dict(self) -> Dict:
        """辞書形式に変換"""
        return {k: v for k, v in asdict(self).items() if v is not None}

@dataclass
class ExtractedData:
    """抽出されたデータ全体"""
    年度: str
    会社名: str
    財務指標: FinancialMetrics
    セグメント情報: Dict[str, str]
    その他の数値: Dict[str, str]

    def to_dict(self) -> Dict:
        """辞書形式に変換"""
        return {
            "年度": self.年度,
            "会社名": self.会社名,
            "財務指標": self.財務指標.to_dict(),
            "セグメント情報": self.セグメント情報,
            "その他の数値": self.その他の数値
        }

# ============================================================================
# 4. PDFテキスト抽出クラス
# ============================================================================

class PDFExtractor:
    """PDFからテキストを抽出"""

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.reader = None

    def extract_text(self, max_pages: int = 50) -> str:
        """PDFからテキストを抽出（最初のmax_pagesページ）"""
        try:
            self.reader = PdfReader(self.pdf_path)
            total_pages = len(self.reader.pages)
            pages_to_extract = min(max_pages, total_pages)

            print(f"PDF総ページ数: {total_pages}")
            print(f"抽出ページ数: {pages_to_extract}")

            text = ""
            for i in range(pages_to_extract):
                page = self.reader.pages[i]
                text += page.extract_text() + "\n"

                if (i + 1) % 10 == 0:
                    print(f"進捗: {i + 1}/{pages_to_extract} ページ")

            return text

        except Exception as e:
            print(f"PDFテキスト抽出エラー: {e}")
            return ""

# ============================================================================
# 5. 数値データ抽出クラス
# ============================================================================

class NumericalDataExtractor:
    """正規表現を使って数値データを抽出"""

    def __init__(self):
        # 数値パターン（カンマ区切り、小数点対応）
        self.number_pattern = r'[\d,]+(?:\.\d+)?'

    def extract_financial_metrics(self, text: str) -> FinancialMetrics:
        """主要な財務指標を抽出"""
        metrics = FinancialMetrics()

        # 営業収益の抽出
        営業収益_pattern = rf'営業収益.*?(?:百万円|円).*?({self.number_pattern})'
        match = re.search(営業収益_pattern, text)
        if match:
            metrics.営業収益 = match.group(1)

        # 税引前利益
        税引前利益_pattern = rf'税引前利益.*?(?:百万円|円).*?({self.number_pattern})'
        match = re.search(税引前利益_pattern, text)
        if match:
            metrics.税引前利益 = match.group(1)

        # 当期純利益（親会社の所有者に帰属）
        当期純利益_pattern = rf'(?:親会社.*?)?当期(?:純)?利益.*?(?:百万円|円).*?({self.number_pattern})'
        match = re.search(当期純利益_pattern, text)
        if match:
            metrics.当期純利益 = match.group(1)

        # 総資産
        総資産_pattern = rf'総資産.*?(?:百万円|円).*?({self.number_pattern})'
        match = re.search(総資産_pattern, text)
        if match:
            metrics.総資産 = match.group(1)

        # 純資産
        純資産_pattern = rf'(?:純資産|親会社.*?持分).*?(?:百万円|円).*?({self.number_pattern})'
        match = re.search(純資産_pattern, text)
        if match:
            metrics.純資産 = match.group(1)

        # ROE
        ROE_pattern = rf'(?:ROE|自己資本.*?利益率).*?[（(]％[)）].*?({self.number_pattern})'
        match = re.search(ROE_pattern, text)
        if match:
            metrics.ROE = match.group(1) + "%"

        # 自己資本比率
        自己資本比率_pattern = rf'(?:自己資本比率|株主資本比率).*?[（(]％[)）].*?({self.number_pattern})'
        match = re.search(自己資本比率_pattern, text)
        if match:
            metrics.自己資本比率 = match.group(1) + "%"

        # 従業員数
        従業員数_pattern = rf'従業員数.*?[（(]人[)）].*?({self.number_pattern})'
        match = re.search(従業員数_pattern, text)
        if match:
            metrics.従業員数 = match.group(1) + "人"

        return metrics

    def extract_segment_info(self, text: str) -> Dict[str, str]:
        """セグメント情報を抽出"""
        segments = {}

        # 自動車セグメント
        auto_pattern = rf'自動車(?:事業)?.*?営業収益.*?({self.number_pattern}).*?百万円'
        match = re.search(auto_pattern, text)
        if match:
            segments["自動車事業営業収益"] = match.group(1) + " 百万円"

        return segments

    def extract_year_and_company(self, text: str) -> Tuple[str, str]:
        """年度と会社名を抽出"""
        year = "不明"
        company = "不明"

        # 年度の抽出
        year_pattern = r'(20\d{2})年[３3]月期'
        match = re.search(year_pattern, text[:1000])
        if match:
            year = f"{match.group(1)}年3月期"

        # 会社名の抽出
        company_pattern = r'【会社名】\s*(.+?)(?:\n|【)'
        match = re.search(company_pattern, text[:2000])
        if match:
            company = match.group(1).strip()

        return year, company

    def extract_all_data(self, text: str) -> ExtractedData:
        """すべてのデータを抽出"""
        year, company = self.extract_year_and_company(text)
        financial_metrics = self.extract_financial_metrics(text)
        segment_info = self.extract_segment_info(text)

        return ExtractedData(
            年度=year,
            会社名=company,
            財務指標=financial_metrics,
            セグメント情報=segment_info,
            その他の数値={}
        )

# ============================================================================
# 6. RAGシステムクラス
# ============================================================================

class RAGSystem:
    """RAGを使った文書検索システム"""

    def __init__(self, embedding_model: str = "intfloat/multilingual-e5-small"):
        """
        Args:
            embedding_model: 使用する埋め込みモデル
        """
        print("埋め込みモデルを読み込み中...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model,
            model_kwargs={'device': 'cpu'}  # Colab無料版用
        )
        self.vectorstore = None
        print("埋め込みモデル読み込み完了")

    def create_vectorstore(self, text: str, chunk_size: int = 500, chunk_overlap: int = 50):
        """テキストからベクトルストアを作成"""
        print("テキストをチャンクに分割中...")

        # テキスト分割
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", "。", "、", " ", ""]
        )

        chunks = text_splitter.split_text(text)
        documents = [Document(page_content=chunk) for chunk in chunks]

        print(f"チャンク数: {len(documents)}")
        print("ベクトルストアを作成中...")

        # ベクトルストア作成
        self.vectorstore = FAISS.from_documents(documents, self.embeddings)
        print("ベクトルストア作成完了")

    def search(self, query: str, k: int = 3) -> List[str]:
        """クエリに関連する文書を検索"""
        if self.vectorstore is None:
            return []

        docs = self.vectorstore.similarity_search(query, k=k)
        return [doc.page_content for doc in docs]

# ============================================================================
# 7. Llama3.2-3Bモデルクラス
# ============================================================================

class Llama3Summarizer:
    """Llama3.2-3Bを使った要約システム"""

    def __init__(self, model_name: str = "meta-llama/Llama-3.2-3B-Instruct"):
        """
        Args:
            model_name: 使用する埋め込みモデル
        """
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.pipeline = None

    def load_model(self):
        """モデルを読み込み（4bit量子化でVRAM節約）"""
        print(f"モデル {self.model_name} を読み込み中...")

        # 4bit量子化設定
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        # トークナイザー読み込み
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        # モデル読み込み
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )

        # パイプライン作成
        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
        )

        print("モデル読み込み完了")

    def summarize_with_context(self, context: str, numerical_data: Dict) -> str:
        """RAGコンテキストと数値データを使って要約"""

        # 数値データをJSON形式で整理
        numerical_json = json.dumps(numerical_data, ensure_ascii=False, indent=2)

        # プロンプト作成
        prompt = f"""以下は有価証券報告書の一部です。重要な情報を簡潔に要約してください。

数値データ（これらの数値は正確なので変更しないでください）:
{numerical_json}

文書の内容:
{context}

上記の情報をもとに、以下の点に注意して要約を作成してください:
1. 数値データはそのまま使用し、変更しないこと
2. 重要なポイントを3-5点にまとめること
3. 簡潔で分かりやすい日本語で記述すること

要約:"""

        messages = [
            {"role": "user", "content": prompt}
        ]

        # 生成
        outputs = self.pipeline(
            messages,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.3,
        )

        # 結果を抽出
        generated_text = outputs[0]["generated_text"]

        # ユーザープロンプト以降の部分を抽出
        if isinstance(generated_text, list):
            # メッセージ形式の場合
            summary = generated_text[-1]["content"]
        else:
            # テキスト形式の場合
            summary = generated_text.split("要約:")[-1].strip()

        return summary

# ============================================================================
# 8. メイン処理クラス
# ============================================================================

class FinancialReportProcessor:
    """有価証券報告書の処理を統合"""

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.pdf_extractor = PDFExtractor(pdf_path)
        self.numerical_extractor = NumericalDataExtractor()
        self.rag_system = None
        self.summarizer = None
        self.extracted_data = None
        self.original_text = ""

    def process(self, max_pages: int = 50):
        """処理を実行"""

        # 1. PDFからテキスト抽出
        print("\n" + "="*60)
        print("ステップ1: PDFからテキストを抽出")
        print("="*60)
        self.original_text = self.pdf_extractor.extract_text(max_pages)
        print(f"抽出したテキスト長: {len(self.original_text)} 文字")

        # 2. 数値データ抽出
        print("\n" + "="*60)
        print("ステップ2: 数値データを抽出")
        print("="*60)
        self.extracted_data = self.numerical_extractor.extract_all_data(self.original_text)
        print(json.dumps(self.extracted_data.to_dict(), ensure_ascii=False, indent=2))

        # 3. RAGシステム初期化
        print("\n" + "="*60)
        print("ステップ3: RAGシステムを初期化")
        print("="*60)
        self.rag_system = RAGSystem()
        self.rag_system.create_vectorstore(self.original_text)

        # 4. Llamaモデル読み込み
        print("\n" + "="*60)
        print("ステップ4: Llama3.2-3Bモデルを読み込み")
        print("="*60)
        self.summarizer = Llama3Summarizer()
        self.summarizer.load_model()

        print("\n処理完了！")

    def generate_summary(self, query: str = "会社の業績と主要な財務指標について") -> str:
        """要約を生成"""

        if self.rag_system is None or self.summarizer is None:
            print("エラー: まずprocess()を実行してください")
            return ""

        print(f"\nクエリ: {query}")
        print("関連文書を検索中...")

        # RAGで関連文書を検索
        contexts = self.rag_system.search(query, k=3)
        combined_context = "\n\n".join(contexts)

        print(f"検索結果: {len(contexts)} 件の関連文書")
        print("\n要約を生成中...")

        # 要約生成
        summary = self.summarizer.summarize_with_context(
            combined_context,
            self.extracted_data.to_dict()
        )

        return summary

    def save_results(self, summary: str, output_dir: str = "./output"):
        """結果を保存"""
        os.makedirs(output_dir, exist_ok=True)

        # JSONで数値データを保存
        json_path = os.path.join(output_dir, "extracted_numbers.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(self.extracted_data.to_dict(), f, ensure_ascii=False, indent=2)
        print(f"数値データを保存: {json_path}")

        # 要約をテキストファイルで保存
        summary_path = os.path.join(output_dir, "summary.txt")
        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(summary)
        print(f"要約を保存: {summary_path}")

        # Excelで保存
        excel_path = os.path.join(output_dir, "financial_data.xlsx")
        df = pd.DataFrame([self.extracted_data.財務指標.to_dict()])
        df.to_excel(excel_path, index=False)
        print(f"財務データ(Excel)を保存: {excel_path}")

# ============================================================================
# 9. 実行例
# ============================================================================

def main():
    """メイン実行関数"""

    # PDFファイルのパス（Google Driveにマウントする場合）
    # from google.colab import drive
    # drive.mount('/content/drive')
    # pdf_path = "/content/drive/MyDrive/your_file.pdf"

    # アップロードしたファイルを使う場合
    from google.colab import files
    print("PDFファイルをアップロードしてください...")
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]

    # 処理実行
    processor = FinancialReportProcessor(pdf_path)
    processor.process(max_pages=50)  # 最初の50ページを処理

    # 要約生成
    summary = processor.generate_summary("会社の業績と主要な財務指標、事業の状況について教えてください")

    print("\n" + "="*60)
    print("生成された要約")
    print("="*60)
    print(summary)

    # 結果を保存
    processor.save_results(summary)

    # 結果をダウンロード
    files.download("output/extracted_numbers.json")
    files.download("output/summary.txt")
    files.download("output/financial_data.xlsx")

# Colab上で実行する場合
if __name__ == "__main__":
    main()


PDFファイルをアップロードしてください...


Saving トヨタ_2025_03_有価証券報告書.pdf to トヨタ_2025_03_有価証券報告書 (1).pdf

ステップ1: PDFからテキストを抽出
PDF総ページ数: 243
抽出ページ数: 50
進捗: 10/50 ページ
進捗: 20/50 ページ
進捗: 30/50 ページ
進捗: 40/50 ページ
進捗: 50/50 ページ
抽出したテキスト長: 54723 文字

ステップ2: 数値データを抽出
{
  "年度": "2025年3月期",
  "会社名": "不明",
  "財務指標": {
    "営業収益": "27,214,594",
    "税引前利益": "2,932,354",
    "当期純利益": "2,245,261",
    "総資産": "62,267,140",
    "純資産": "13,894,021",
    "ROE": "12.4%",
    "自己資本比率": "65.5%"
  },
  "セグメント情報": {},
  "その他の数値": {}
}

ステップ3: RAGシステムを初期化
埋め込みモデルを読み込み中...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


埋め込みモデル読み込み完了
テキストをチャンクに分割中...
チャンク数: 120
ベクトルストアを作成中...
ベクトルストア作成完了

ステップ4: Llama3.2-3Bモデルを読み込み
モデル meta-llama/Llama-3.2-3B-Instruct を読み込み中...


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>